# Adaptive Commodity Allocation in 5 Minutes

**Goal:** Use Thompson Sampling to dynamically allocate capital across commodity sectors.

**What you'll learn:** How to frame portfolio allocation as a bandit problem with Gaussian rewards.

**Scenario:** You have capital to allocate across Energy, Metals, and Agriculture. Which sector should get more weight each week?

---

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
!pip install -q numpy pandas matplotlib yfinance scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
apply_course_theme()
apply_plot_theme()

## Load Commodity Sector Data

We'll try to load real ETF data. If it fails, we'll use synthetic data.

In [ ]:
# Try to load real data from Yahoo Finance
try:
    import yfinance as yf
    
    # Commodity sector proxies
    tickers = {
        'Energy': 'XLE',      # Energy Select Sector SPDR Fund
        'Metals': 'XME',      # SPDR S&P Metals & Mining ETF
        'Agriculture': 'DBA'  # Invesco DB Agriculture Fund
    }
    
    print("Downloading commodity data...")
    data = yf.download(list(tickers.values()), period='2y', interval='1wk', progress=False)['Adj Close']
    data.columns = list(tickers.keys())
    
    # Calculate weekly returns
    returns = data.pct_change().dropna()
    
    print(f"✓ Loaded {len(returns)} weeks of real commodity data")
    print(f"  Date range: {returns.index[0].date()} to {returns.index[-1].date()}")
    
except Exception as e:
    print(f"Could not load real data: {e}")
    print("\nUsing synthetic commodity returns instead...")
    
    # Generate synthetic returns with realistic parameters
    n_weeks = 104  # 2 years
    returns = pd.DataFrame({
        'Energy': np.random.normal(0.002, 0.04, n_weeks),
        'Metals': np.random.normal(0.001, 0.03, n_weeks),
        'Agriculture': np.random.normal(0.0015, 0.035, n_weeks)
    })
    print(f"✓ Generated {len(returns)} weeks of synthetic data")

print(f"\nAverage weekly returns:")
print(returns.mean() * 100)

## Frame as a Bandit Problem

**Arms:** 3 commodity sectors

**Rewards:** Weekly returns (Gaussian distributed)

**Goal:** Allocate capital to maximize cumulative returns over 52 weeks

**Algorithm:** Thompson Sampling with Gaussian priors

In [ ]:
n_arms = 3
sectors = list(returns.columns)

print("Bandit setup:")
print(f"  Arms: {sectors}")
print(f"  Rewards: Weekly returns")
print(f"  Horizon: 52 weeks")
print(f"\nStrategy: Each week, pick the sector that looks most promising")

## Thompson Sampling with Gaussian Rewards

For continuous rewards, we track mean and precision (inverse variance) using a Normal-Gamma prior.

In [ ]:
# Use first 52 weeks for bandit simulation
n_weeks = min(52, len(returns))
returns_sample = returns.iloc[:n_weeks]

# Thompson Sampling with Gaussian priors
# Track sample mean and count for each arm
sum_rewards = np.zeros(n_arms)
sum_squared_rewards = np.zeros(n_arms)
n_pulls = np.zeros(n_arms)

# History tracking
choices = []
rewards_obtained = []
cumulative_returns = [1.0]  # Start with $1

for week in range(n_weeks):
    # Sample from posterior for each arm
    theta_samples = []
    
    for arm in range(n_arms):
        if n_pulls[arm] == 0:
            # No data yet, sample from prior N(0, 0.1^2)
            theta = np.random.normal(0, 0.1)
        else:
            # Sample from posterior
            sample_mean = sum_rewards[arm] / n_pulls[arm]
            sample_var = max(1e-6, (sum_squared_rewards[arm] / n_pulls[arm] - sample_mean**2))
            sample_std = np.sqrt(sample_var / n_pulls[arm])
            theta = np.random.normal(sample_mean, sample_std)
        
        theta_samples.append(theta)
    
    # Choose arm with highest sample
    chosen_arm = np.argmax(theta_samples)
    choices.append(chosen_arm)
    
    # Observe reward (actual weekly return)
    reward = returns_sample.iloc[week, chosen_arm]
    rewards_obtained.append(reward)
    
    # Update statistics
    n_pulls[chosen_arm] += 1
    sum_rewards[chosen_arm] += reward
    sum_squared_rewards[chosen_arm] += reward ** 2
    
    # Track cumulative return
    cumulative_returns.append(cumulative_returns[-1] * (1 + reward))

print(f"Completed {n_weeks} weeks of adaptive allocation")

## Compare to Equal-Weight Allocation

Baseline: invest equally in all three sectors every week.

In [ ]:
# Equal-weight baseline: 1/3 in each sector
equal_weight_returns = returns_sample.mean(axis=1)
equal_weight_cumulative = [1.0]

for r in equal_weight_returns:
    equal_weight_cumulative.append(equal_weight_cumulative[-1] * (1 + r))

# Results
bandit_final = cumulative_returns[-1]
equal_final = equal_weight_cumulative[-1]

print("\n=== Performance Comparison ===")
print(f"\nThompson Sampling:")
print(f"  Final portfolio value: ${bandit_final:.3f}")
print(f"  Total return: {100*(bandit_final - 1):.2f}%")

print(f"\nEqual-Weight:")
print(f"  Final portfolio value: ${equal_final:.3f}")
print(f"  Total return: {100*(equal_final - 1):.2f}%")

print(f"\nDifference: {100*(bandit_final - equal_final):.2f}% {'outperformance' if bandit_final > equal_final else 'underperformance'}")

## Visualize Results

Plot cumulative returns and sector allocation over time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Cumulative returns
ax1.plot(cumulative_returns, label='Thompson Sampling', linewidth=2.5, alpha=0.8)
ax1.plot(equal_weight_cumulative, label='Equal-Weight', linewidth=2.5, alpha=0.8, linestyle='--')
ax1.set_xlabel('Week', fontsize=12)
ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
ax1.set_title('Cumulative Returns: Adaptive vs Equal-Weight', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.axhline(1.0, color='black', linestyle=':', alpha=0.5)

# Plot 2: Allocation weights over time
for arm in range(n_arms):
    # Calculate rolling allocation (% of time each arm was chosen in last 10 weeks)
    window = 10
    allocation_pct = []
    for i in range(len(choices)):
        start = max(0, i - window)
        window_choices = choices[start:i+1]
        pct = 100 * window_choices.count(arm) / len(window_choices)
        allocation_pct.append(pct)
    
    ax2.plot(allocation_pct, label=sectors[arm], linewidth=2, alpha=0.8)

ax2.axhline(33.33, color='black', linestyle='--', alpha=0.3, label='Equal-weight (33.3%)')
ax2.set_xlabel('Week', fontsize=12)
ax2.set_ylabel('Allocation % (10-week rolling)', fontsize=12)
ax2.set_title('Thompson Sampling: Sector Allocation Over Time', fontsize=14)
ax2.set_ylim([0, 100])
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNotice: The bandit shifts allocation toward better-performing sectors!")

## Sector Selection Summary

How often was each sector chosen?

In [ ]:
print("\n=== Sector Selection Frequency ===")
for arm in range(n_arms):
    count = choices.count(arm)
    pct = 100 * count / len(choices)
    avg_return = sum_rewards[arm] / n_pulls[arm] if n_pulls[arm] > 0 else 0
    print(f"{sectors[arm]:12} : {count:2d} weeks ({pct:5.1f}%) - Avg return: {100*avg_return:+.2f}%/week")

print(f"\nTotal weeks: {len(choices)}")

# Compare to true average returns
print("\n=== True Average Returns (all data) ===")
for sector in sectors:
    true_avg = returns_sample[sector].mean()
    print(f"{sector:12} : {100*true_avg:+.2f}%/week")

## Modify This

Experiment with different setups:

1. **More sectors:** Add more commodity tickers like 'GLD' (gold), 'USO' (oil), 'CORN'
2. **Longer horizon:** Use `n_weeks = 104` (2 years) - does performance improve?
3. **Transaction costs:** Subtract 0.1% each time you switch sectors
4. **Weighted allocation:** Instead of picking one sector, allocate proportional to Thompson samples
5. **Compare algorithms:** Try UCB1 or ε-greedy instead of Thompson Sampling

In [ ]:
# YOUR EXPERIMENTS HERE
# Try different tickers, horizons, or algorithms

## What's Next?

You just applied Thompson Sampling to real commodity data!

**Continue learning:**
- `00_your_first_bandit.ipynb` - Thompson Sampling fundamentals
- `01_ab_test_vs_bandit.ipynb` - Why bandits beat A/B tests
- `04_algorithm_comparison.ipynb` - Compare different bandit algorithms

**Deep dive:**
- **Module 2:** Gaussian bandits and conjugate priors
- **Module 5:** The Two-Wallet Framework for commodity allocation (production-ready)
- **Module 6:** Non-stationary bandits (market regimes change!)
- **Template:** `commodity_allocation_template.py` - Production code with risk controls

**Key insight:** This is a simplified single-arm allocation. Module 5 covers the full two-wallet framework with exploration and exploitation budgets!